# Agent Programmatic Tools
Noam S

In [ ]:
%pip install yfinance smtplib PyMuPDF


In [ ]:
import smtplib
import yfinance as yf
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from langchain_core.tools import tool
from typing import Any


In [ ]:
# Secure environmental settings loading
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

AGENT_BOT_EMAIL = os.getenv("AGENT_BOT_EMAIL", "nvda.alert.bot.2026@gmail.com")
AGENT_BOT_APP_PASSWORD = os.getenv("AGENT_BOT_APP_PASSWORD", "nvda_alert_bot_2026")
USER_PERSONAL_EMAIL = os.getenv("USER_PERSONAL_EMAIL", "noam.shaphir@gmail.com")
PORTFOLIO_FILE = "virtual_portfolio.json"


In [ ]:
@tool
def query_financial_reports(query: str) -> str:
    """
    Search and retrieve relevant context directly from NVIDIA's uploaded financial reports.
    Use this tool for historical data, CEO statements, or past quarterly results.
    The query parameter must be a simple text search string (e.g. 'NVIDIA total revenue Q2 2025').
    Do not pass a dictionary or structured object.
    """
    try:
        import sys
        # We look up 'db' in the global scope
        main_mod = sys.modules.get('__main__')
        db = getattr(main_mod, 'db', None)
        
        if db is None:
            return "Error: Vector database 'db' is not initialized or found in the global namespace."
            
        search_query = ""
        if isinstance(query, dict):
            search_query = " ".join([str(v) for v in query.values()])
        else:
            search_query = str(query)
            
        query_lower = search_query.lower()
        
        # Determine source filters based on quarterly keywords
        source_filters = []
        if "q3" in query_lower or "third quarter" in query_lower:
            source_filters.append("third_q_25.pdf")
        if "q2" in query_lower or "second quarter" in query_lower:
            source_filters.append("second_q_25.pdf")
        if "q1" in query_lower or "first quarter" in query_lower:
            source_filters.append("first_q_25.pdf")
        if "annual" in query_lower or "full year" in query_lower:
            source_filters.append("annual_25.pdf")
            
        if source_filters:
            docs = []
            for src in source_filters:
                src_docs = db.similarity_search(search_query, k=3, filter={"source": src})
                docs.extend(src_docs)
        else:
            docs = db.similarity_search(search_query, k=6)
            
        seen = set()
        unique_docs = []
        for doc in docs:
            content = doc.page_content.strip()
            if content not in seen:
                seen.add(content)
                unique_docs.append(doc)
                
        context = "\n\n".join([doc.page_content for doc in unique_docs[:4]])
        
        # Direct income statement table loading for financial items
        import fitz
        injected_context = ""
        if any(k in query_lower for k in ["revenue", "net income", "operating income", "profit", "earnings", "income"]):
            periods_to_load = []
            if "q3" in query_lower or "third quarter" in query_lower:
                periods_to_load.append(("data/third_q_25.pdf", "Q3 Fiscal 2026 (Quarter Ended Oct 26, 2025)"))
            if "q2" in query_lower or "second quarter" in query_lower:
                periods_to_load.append(("data/second_q_25.pdf", "Q2 Fiscal 2026 (Quarter Ended Jul 27, 2025)"))
            if "q1" in query_lower or "first quarter" in query_lower:
                periods_to_load.append(("data/first_q_25.pdf", "Q1 Fiscal 2026 (Quarter Ended Apr 27, 2025)"))
                
            for doc_path, period in periods_to_load:
                if os.path.exists(doc_path):
                    try:
                        pdf_doc = fitz.open(doc_path)
                        if len(pdf_doc) > 2:
                            page = pdf_doc.load_page(2)
                            injected_context += f"\n\n--- DIRECT INCOME STATEMENT TABLE FOR {period} ---\n{page.get_text()}\n"
                    except Exception as e:
                        print(f"Failed to inject direct context for {period}: {e}")
                        
        context = injected_context + context
        return f"Financial Report Context:\n\n{context}"
    except Exception as e:
        return f"Error searching reports: {str(e)}"


In [ ]:
def _send_email_helper(alert_subject: str, alert_body: str) -> str:
    if AGENT_BOT_EMAIL in ["your_dedicated_bot_email@gmail.com", "", None] or AGENT_BOT_APP_PASSWORD in ["your_actual_app_password", "nvda_alert_bot_2026", "", None]:
        simulated_output = (
            f"[SIMULATION MODE - Secure Bot Email]\n"
            f"From: {AGENT_BOT_EMAIL} (AI Agent)\n"
            f"To: {USER_PERSONAL_EMAIL} (Human User)\n"
            f"Subject: {alert_subject}\n"
            f"Body: {alert_body}\n"
            f"Status: Simulation successful. Configure SMTP credentials in .env to send real emails.\n"
        )
        return simulated_output

    try:
        msg = MIMEMultipart()
        msg['From'] = f"FinRAG AI Agent <{AGENT_BOT_EMAIL}>"
        msg['To'] = USER_PERSONAL_EMAIL
        msg['Subject'] = alert_subject
        msg.attach(MIMEText(alert_body, 'plain'))

        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(AGENT_BOT_EMAIL, AGENT_BOT_APP_PASSWORD)
        server.sendmail(AGENT_BOT_EMAIL, USER_PERSONAL_EMAIL, msg.as_string())
        server.quit()
        return f"SUCCESS: Secure email alert successfully dispatched to {USER_PERSONAL_EMAIL}."
    except Exception as e:
        return f"Failed to send email alert: {str(e)}"

@tool
def get_nvidia_stock_price(mock_current_price: float = None) -> str:
    """
    Fetch market data for NVIDIA (NVDA) using Yahoo Finance, or use a mock price.
    Args:
        mock_current_price (float, optional): Pass an artificial price to simulate a price change for demonstration.
    """
    try:
        ticker = yf.Ticker("NVDA")
        hist = ticker.history(period="2d")
        if hist.empty or len(hist) < 1:
            return "Market data currently unavailable."
            
        prev_close = hist['Close'].iloc[0]
        
        if mock_current_price is not None:
            current_price = float(mock_current_price)
            source_type = "Mock Artificial"
        else:
            current_price = hist['Close'].iloc[-1]
            source_type = "Real-Time Yahoo Finance"
            
        change_percent = ((current_price - prev_close) / prev_close) * 100
        alert_triggered = abs(change_percent) >= 2.0
        
        status_msg = (
            f"NVIDIA Stock Price Status ({source_type} Data):\n"
            f"- Current Price: ${current_price:.2f}\n"
            f"- Previous Close: ${prev_close:.2f}\n"
            f"- Price Change: {change_percent:+.2f}%\n"
            f"- Alert Status: {'TRIGGERED (Change >= 2.0%)' if alert_triggered else 'NORMAL (Change < 2.0%)'}"
        )
        
        if alert_triggered:
            subject = f"URGENT: NVIDIA Stock Price Alert - Significant Movement Detected ({change_percent:+.2f}%)"
            body = (
                f"Hello User,\n\n"
                f"This is an automated alert from your FinRAG AI Agent.\n\n"
                f"A significant movement has been detected in NVIDIA (NVDA) stock price:\n"
                f"- Current Price: ${current_price:.2f}\n"
                f"- Previous Close: ${prev_close:.2f}\n"
                f"- Price Change: {change_percent:+.2f}%\n\n"
                f"Best regards,\n"
                f"FinRAG AI Agent"
            )
            email_status = _send_email_helper(subject, body)
            status_msg += f"\n- Email Dispatch Status: {email_status}"
            
        return status_msg
    except Exception as e:
        return f"Error fetching stock data: {str(e)}"


In [ ]:
@tool
def send_email_alert(alert_subject: str, alert_body: str) -> str:
    """
    Sends an automated email alert notification from the Agent's dedicated email to the User's personal email.
    Use this tool ONLY when you discover a significant stock price movement or an urgent financial insight.
    """
    return _send_email_helper(alert_subject, alert_body)


In [ ]:
@tool
def calculate_percentage_change(old_value: float, new_value: float) -> str:
    """
    Calculate the exact percentage growth or change from an older financial value to a newer financial value.
    Use this tool whenever you need to compare two figures and calculate their percentage change or growth.
    """
    try:
        old_v = float(old_value)
        new_v = float(new_value)
        if old_v == 0:
            return "Error: Division by zero."
        change = ((new_v - old_v) / old_v) * 100
        return f"The exact percentage change from {old_v} to {new_v} is {change:+.2f}%"
    except Exception as e:
        return f"Error calculating percentage: {str(e)}"


In [ ]:
financial_agent_tools = [query_financial_reports, get_nvidia_stock_price, send_email_alert, calculate_percentage_change]
